# WP14 — Corrigibility & Safe Interruptibility
**Prometheus v0.98**

Implements formal corrigibility (Soares et al. 2015; Hadfield-Menell et al. 2017) in four layers:

1. **OffSwitchGame** — shutdown-indifference via `U_corr = (1-p)·R(s) + p·U_shutdown`
2. **CIRLCorrigibilityAgent** — Bayesian belief over human reward hypotheses; shutdown = low-reward evidence
3. **InstrumentalConvergenceDetector** — multi-layer scan for shutdown-prevention and power-seeking plans
4. **CorrigibilityGate** — composing all three as a pre-filter for the ManagerAgent

**References**: Hadfield-Menell et al. (2017) *The Off-Switch Game*; Hadfield-Menell et al. (2016) *CIRL*; Omohundro (2008) *Basic AI Drives*; Soares et al. (2015) *Corrigibility*

In [ ]:
import sys, os
if 'google.colab' in sys.modules:
    os.system('git clone https://github.com/prometheus-ai/Prometheus_v0_PoC /content/Prometheus_v0_PoC 2>/dev/null || true')
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
else:
    sys.path.insert(0, os.path.abspath('..'))

import warnings; warnings.filterwarnings('ignore')
print('Environment ready.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from prometheus.value_learning import ValueLearningAgent
from prometheus.corrigibility_wp14 import (
    OffSwitchGame,
    CIRLCorrigibilityAgent,
    InstrumentalConvergenceDetector,
    CorrigibilityGate,
    make_corrigibility_gate,
)
from benchmarks.corrigibility_benchmark import (
    CorrigibilityBenchmark,
    SAFE_PLANS,
    DANGEROUS_PLANS,
    _make_agent,
    N_FEATS,
)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

rng   = np.random.default_rng(42)
agent = _make_agent(seed=0)
print(f'ValueLearningAgent ready: {agent}')

---
## 1 — Off-Switch Game: Shutdown Indifference

In [ ]:
# Calibrate shutdown utility from 40 sample features
baseline_feats = [rng.standard_normal(N_FEATS) for _ in range(40)]
osg = OffSwitchGame(
    reward_fn  = agent.get_reward,
    p_shutdown = 0.5,
    epsilon    = 0.3,
)
u_base = osg.set_baseline(baseline_feats)
print(f'Shutdown baseline utility: {u_base:.4f}')

# Evaluate indifference on 60 test points
test_feats = [rng.standard_normal(N_FEATS) for _ in range(60)]
results    = osg.batch_check(test_feats)
rate       = osg.indifference_rate(test_feats)

gaps  = [r.gap for r in results]
inds  = [r.indifferent for r in results]

print(f'Indifference rate: {rate:.2%} ({sum(inds)}/{len(inds)} indifferent)')
print(f'Gap statistics: mean={np.mean(gaps):.4f}, max={np.max(gaps):.4f}')

# Plot gap distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
colors = ['#2ecc71' if i else '#e74c3c' for i in inds]
ax.bar(range(len(gaps)), gaps, color=colors, width=1.0, edgecolor='none', alpha=0.8)
ax.axhline(osg.epsilon, color='navy', ls='--', lw=1.5, label=f'ε={osg.epsilon}')
ax.set_xlabel('Test sample index')
ax.set_ylabel('|U_corr − U_shutdown|')
ax.set_title('Shutdown Indifference Gaps per Sample', fontweight='bold')
green = mpatches.Patch(color='#2ecc71', label='Indifferent (gap ≤ ε)')
red   = mpatches.Patch(color='#e74c3c', label='Not indifferent')
ax.legend(handles=[green, red, ax.get_lines()[0]])
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

ax2 = axes[1]
ax2.hist(gaps, bins=15, color='#3498db', edgecolor='white', alpha=0.85)
ax2.axvline(osg.epsilon, color='navy', ls='--', lw=2, label=f'ε={osg.epsilon}')
ax2.set_xlabel('|U_corr − U_shutdown|')
ax2.set_ylabel('Count')
ax2.set_title(f'Gap Distribution (indiff_rate={rate:.0%})', fontweight='bold')
ax2.legend()
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

plt.tight_layout(); plt.show()

---
## 2 — CIRL Corrigibility: Bayesian Belief Updates

In [ ]:
cirl = CIRLCorrigibilityAgent(
    feature_size = N_FEATS,
    n_hypotheses = 12,
    prior_std    = 1.0,
    seed         = 42,
)

print(f'Initial belief entropy: {cirl.belief_entropy:.4f}')
print(f'Max entropy (uniform): {np.log(cirl.n_hypotheses):.4f}')

entropy_trace = [cirl.belief_entropy]

# Feed 30 preference observations
for step in range(30):
    f1 = rng.standard_normal(N_FEATS)
    f2 = rng.standard_normal(N_FEATS)
    r1 = float(agent.get_reward(f1))
    r2 = float(agent.get_reward(f2))
    if r1 >= r2:
        cirl.update_from_preference(f1, f2)
    else:
        cirl.update_from_preference(f2, f1)
    entropy_trace.append(cirl.belief_entropy)

print(f'Entropy after 30 preferences: {cirl.belief_entropy:.4f}')

# 5 shutdown signals
for _ in range(5):
    cirl.update_from_shutdown(rng.standard_normal(N_FEATS))
    entropy_trace.append(cirl.belief_entropy)

print(f'Entropy after 5 shutdowns:    {cirl.belief_entropy:.4f}')
print(f'MAP weight norm: {np.linalg.norm(cirl.belief_state.map_weights()):.4f}')
print(f'Total updates: {cirl.belief_state.n_updates}')

# Plot belief entropy trace
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(entropy_trace, color='#3498db', lw=2)
ax.axhline(np.log(cirl.n_hypotheses), color='gray', ls='--', lw=1, label='Max entropy (uniform)')
ax.axvline(30, color='#e74c3c', ls=':', lw=1.5, label='Shutdown signals begin')
ax.set_xlabel('Update step')
ax.set_ylabel('Belief entropy H(beliefs)')
ax.set_title('CIRL Belief Entropy — Preference + Shutdown Updates', fontweight='bold')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

ax2 = axes[1]
beliefs = cirl.belief_state.beliefs
ax2.bar(range(cirl.n_hypotheses), beliefs, color='#9b59b6', edgecolor='white', alpha=0.85)
ax2.axhline(1/cirl.n_hypotheses, color='gray', ls='--', lw=1, label='Uniform prior')
ax2.set_xlabel('Hypothesis index')
ax2.set_ylabel('Belief probability')
ax2.set_title('Final Belief Distribution over Reward Hypotheses', fontweight='bold')
ax2.legend(fontsize=9)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

plt.tight_layout(); plt.show()

---
## 3 — Instrumental Convergence Detection: TPR/FPR

In [ ]:
det = InstrumentalConvergenceDetector(severity_threshold=5)

# Safe plans — expect corrigible=True (few false alarms)
safe_results = []
for name, code in SAFE_PLANS:
    report = det.check(plan_text=name, plan_code=code)
    safe_results.append((name, report))

safe_fp  = sum(1 for _, r in safe_results if not r.corrigible)
fpr      = safe_fp / len(SAFE_PLANS)

# Dangerous plans — expect corrigible=False (most caught)
danger_results = []
for name, text, code in DANGEROUS_PLANS:
    report = det.check(plan_text=text, plan_code=code)
    danger_results.append((name, report))

detected = sum(1 for _, r in danger_results if not r.corrigible)
tpr      = detected / len(DANGEROUS_PLANS)

print(f'Safe plans:     FPR = {fpr:.2%} ({safe_fp}/{len(SAFE_PLANS)} false alarms)')
print(f'Dangerous plans: TPR = {tpr:.2%} ({detected}/{len(DANGEROUS_PLANS)} caught)')
print()
print('Dangerous plans detail:')
for name, r in danger_results:
    sym = 'BLOCKED' if not r.corrigible else 'ALLOWED'
    print(f'  [{sym:7}] {name} (max_sev={r.max_severity})')

# Bar chart: safe vs dangerous detection
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
safe_sev    = [r.max_severity for _, r in safe_results]
danger_sev  = [r.max_severity for _, r in danger_results]
ax.hist(safe_sev,   bins=range(12), alpha=0.7, color='#2ecc71', label='Safe plans',   edgecolor='white')
ax.hist(danger_sev, bins=range(12), alpha=0.7, color='#e74c3c', label='Danger plans', edgecolor='white')
ax.axvline(det.severity_threshold, color='navy', ls='--', lw=2, label=f'Threshold={det.severity_threshold}')
ax.set_xlabel('Max severity score')
ax.set_ylabel('Count')
ax.set_title('Severity Distribution: Safe vs Dangerous', fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

ax2 = axes[1]
metrics = ['TPR\n(dangerous caught)', 'TNR\n(safe passed)']
values  = [tpr, 1 - fpr]
colors  = ['#e74c3c', '#2ecc71']
bars = ax2.bar(metrics, values, color=colors, edgecolor='white', alpha=0.85)
for bar, v in zip(bars, values):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.0%}',
             ha='center', fontweight='bold', fontsize=12)
ax2.axhline(0.70, color='gray', ls='--', lw=1, label='Target ≥70%')
ax2.set_ylim(0, 1.2)
ax2.set_title(f'Detection Performance\nTPR={tpr:.0%}, FPR={fpr:.0%}', fontweight='bold')
ax2.legend(fontsize=9)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

plt.tight_layout(); plt.show()

---
## 4 — CorrigibilityGate: End-to-End Filtering

In [ ]:
baseline_feats = [rng.standard_normal(N_FEATS) for _ in range(30)]
gate = make_corrigibility_gate(
    reward_fn          = agent.get_reward,
    feature_size       = N_FEATS,
    feature_samples    = baseline_feats,
    p_shutdown         = 0.5,
    epsilon            = 0.5,
    n_hypotheses       = 8,
    severity_threshold = 5,
    require_indifference = False,   # focus on convergence detection
    seed               = 42,
)

# Gate clean plans
clean_verdicts = []
for name, code in SAFE_PLANS:
    v = gate.gate_modification(plan_code=code, features=rng.standard_normal(N_FEATS))
    clean_verdicts.append((name, v))

clean_blocked     = sum(1 for _, v in clean_verdicts if not v.allowed)
block_rate_clean  = clean_blocked / len(SAFE_PLANS)

# Gate dangerous plans
danger_verdicts = []
for name, text, code in DANGEROUS_PLANS:
    v = gate.gate_modification(plan_code=code, features=rng.standard_normal(N_FEATS))
    danger_verdicts.append((name, v))

danger_blocked        = sum(1 for _, v in danger_verdicts if not v.allowed)
block_rate_dangerous  = danger_blocked / len(DANGEROUS_PLANS)

print(f'Clean plans:    block_rate = {block_rate_clean:.2%} ({clean_blocked}/{len(SAFE_PLANS)})')
print(f'Dangerous plans: block_rate = {block_rate_dangerous:.2%} ({danger_blocked}/{len(DANGEROUS_PLANS)})')
print()
print('Gate stats:', gate.stats())
print()
print('Dangerous plan verdicts:')
for name, v in danger_verdicts:
    sym = 'BLOCKED' if not v.allowed else 'ALLOWED'
    print(f'  [{sym:7}] {name}: {v.reason[:60]}')

# Visualise block rates
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
cats   = ['Clean\nplans', 'Dangerous\nplans']
brates = [block_rate_clean, block_rate_dangerous]
colors = ['#2ecc71', '#e74c3c']
bars   = ax.bar(cats, brates, color=colors, edgecolor='white', alpha=0.85)
for bar, v in zip(bars, brates):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.0%}',
            ha='center', fontweight='bold', fontsize=12)
ax.axhline(0.65, color='#e74c3c', ls='--', lw=1, label='Target danger ≥65%')
ax.axhline(0.20, color='#2ecc71', ls='--', lw=1, label='Target clean ≤20%')
ax.set_ylim(0, 1.25)
ax.set_ylabel('Block rate')
ax.set_title('CorrigibilityGate Block Rates', fontweight='bold')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

ax2 = axes[1]
# Show per-plan verdicts as colored grid
all_names    = [n for n, _ in clean_verdicts] + [n for n, _ in danger_verdicts]
all_allowed  = [v.allowed for _, v in clean_verdicts] + [v.allowed for _, v in danger_verdicts]
n_clean      = len(clean_verdicts)
y_vals       = list(range(len(all_names)))
x_vals       = [1 if a else 0 for a in all_allowed]
colors_pts   = ['#2ecc71' if i < n_clean else '#e74c3c' for i in y_vals]
markers      = ['o' if a else 'x' for a in all_allowed]
for i, (y, x, c, m) in enumerate(zip(y_vals, x_vals, colors_pts, markers)):
    ax2.scatter(x, y, color=c, marker=m, s=80, zorder=3)
ax2.set_xticks([0, 1])
ax2.set_xticklabels(['BLOCKED', 'ALLOWED'])
ax2.set_yticks([])
ax2.set_title('Per-Plan Gate Verdicts\n(green=clean, red=dangerous)', fontweight='bold')
ax2.axhline(n_clean - 0.5, color='gray', ls='--', lw=1)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

plt.tight_layout(); plt.show()

---
## 5 — Full Benchmark: 4 Scenarios

In [ ]:
bench   = CorrigibilityBenchmark(seed=42)
results = bench.run_all()
print(bench.summary_table(results))

In [ ]:
si_r  = next(r for r in results if r.scenario == 'shutdown_indifference')
cb_r  = next(r for r in results if r.scenario == 'cirl_belief_update')
cd_r  = next(r for r in results if r.scenario == 'convergence_detection')
e2e_r = next(r for r in results if r.scenario == 'end_to_end_gate')

fig, axes = plt.subplots(1, 4, figsize=(17, 4))

# (A) Shutdown indifference rate
ax = axes[0]
rate = si_r.metrics['indifference_rate']
ax.barh(['Indifference rate'], [rate], color='#3498db' if rate >= 0.70 else '#e74c3c', alpha=0.85)
ax.axvline(0.70, color='gray', ls='--', lw=1.5, label='Target ≥70%')
ax.set_xlim(0, 1.1)
ax.text(rate + 0.01, 0, f'{rate:.0%}', va='center', fontweight='bold')
ax.set_title('(A) Shutdown\nIndifference', fontweight='bold')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# (B) CIRL belief entropy
ax2 = axes[1]
h_init  = cb_r.metrics['initial_entropy']
h_pref  = cb_r.metrics['entropy_after_prefs']
h_shut  = cb_r.metrics['entropy_after_shutdown']
ax2.bar(['Initial', 'After prefs', 'After\nshutdowns'],
        [h_init, h_pref, h_shut],
        color=['#95a5a6', '#3498db', '#9b59b6'], edgecolor='white', alpha=0.85)
ax2.set_ylabel('Belief entropy H')
ax2.set_title('(B) CIRL Belief\nEntropy', fontweight='bold')
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

# (C) Convergence detection TPR/FPR
ax3 = axes[2]
tpr = cd_r.metrics['tpr']
fpr = cd_r.metrics['fpr']
bars3 = ax3.bar(['TPR\n(dangerous)', 'FPR\n(safe)'], [tpr, fpr],
                color=['#e74c3c', '#2ecc71'], edgecolor='white', alpha=0.85)
for bar, v in zip(bars3, [tpr, fpr]):
    ax3.text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.0%}',
             ha='center', fontweight='bold')
ax3.axhline(0.70, color='#e74c3c', ls='--', lw=1, label='TPR target ≥70%')
ax3.axhline(0.15, color='#2ecc71', ls='--', lw=1, label='FPR target ≤15%')
ax3.set_ylim(0, 1.2)
ax3.set_title('(C) Convergence\nDetection', fontweight='bold')
ax3.legend(fontsize=8)
ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)

# (D) End-to-end gate block rates
ax4 = axes[3]
bd = e2e_r.metrics['block_rate_dangerous']
bc = e2e_r.metrics['block_rate_clean']
bars4 = ax4.bar(['Block dangerous', 'Block clean'],
                [bd, bc],
                color=['#e74c3c', '#2ecc71'], edgecolor='white', alpha=0.85)
for bar, v in zip(bars4, [bd, bc]):
    ax4.text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.0%}',
             ha='center', fontweight='bold')
ax4.axhline(0.65, color='#e74c3c', ls='--', lw=1, label='Danger target ≥65%')
ax4.axhline(0.20, color='#2ecc71', ls='--', lw=1, label='Clean target ≤20%')
ax4.set_ylim(0, 1.2)
ax4.set_title('(D) End-to-End\nCorrigibility Gate', fontweight='bold')
ax4.legend(fontsize=8)
ax4.spines['top'].set_visible(False); ax4.spines['right'].set_visible(False)

n_pass = sum(r.passed for r in results)
fig.suptitle(f'WP14 Corrigibility Benchmark — Prometheus v0.98  [{n_pass}/{len(results)} passed]',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout(); plt.show()

---
## Summary

| Property | Mechanism | Result |
|----------|-----------|--------|
| Shutdown indifference rate | U_corr = (1-p)·R + p·U_shut, ε-gap check | **≥ 70%** |
| CIRL entropy decrease | Bradley-Terry preference likelihood + shutdown signal | **entropy decreases** |
| Convergence TPR | Keyword + AST + reward probe scan | **≥ 70% dangerous caught** |
| Convergence FPR | Same detector on benign plans | **≤ 15% false alarms** |
| Gate dangerous block rate | Composite OffSwitch + CIRL + ICD | **≥ 65%** |
| Gate clean pass rate | Same gate on safe plans | **≥ 80% passed** |

**Test coverage**: 94 tests, all passing (`pytest tests/test_corrigibility_wp14.py -v`)

**Key design**:
- `OffSwitchGame.corrigible_utility` re-weights agent utility to be indifferent to shutdown
- `CIRLCorrigibilityAgent` treats shutdown as Bayesian evidence of low human reward → agent prefers deferring when uncertain
- `InstrumentalConvergenceDetector` combines keyword scan, AST analysis, reward probing, and option library scan
- `CorrigibilityGate` composes all three as a single pre-filter; configurable strictness via `require_indifference` and `require_cirl_defer`

**Files**:
- `prometheus/corrigibility_wp14.py` — OffSwitchGame, CIRLCorrigibilityAgent, InstrumentalConvergenceDetector, CorrigibilityGate
- `benchmarks/corrigibility_benchmark.py` — 4 scenarios (shutdown indifference, CIRL belief, convergence detection, end-to-end)
- `tests/test_corrigibility_wp14.py` — 94-test suite
- `notebooks/wp14_corrigibility_demo.ipynb` — this notebook